In [ ]:
import pandas as pd

raw = pd.read_json("../data/candidates.jsonl", lines=True)

df = pd.json_normalize(
    raw.to_dict(orient="records"),
    sep="_"
)

print(df.dtypes)
print(df.isna().sum())

In [ ]:
df["profile_years_of_experience"].describe()

In [ ]:
def score_experience(years):
    if 5 <= years <= 9:
        return 1.0
    if (4 <= years < 5) or (9 < years <= 10):
        return 0.8
    if (3 <= years < 4) or (10 < years <= 12):
        return 0.5
    return 0.2

df["experience_score"] = df["profile_years_of_experience"].apply(score_experience)

print("Value counts:")
print(df["experience_score"].value_counts())

print("\nDescriptive statistics:")
print(df["experience_score"].describe())

In [ ]:
df["profile_current_title"].value_counts()

In [ ]:
strong_title_matches = {
    "Recommendation Systems Engineer",
    "Search Engineer",
    "Machine Learning Engineer",
    "Senior Machine Learning Engineer",
    "Applied ML Engineer",
    "AI Engineer",
    "Lead AI Engineer",
    "NLP Engineer",
    "Senior NLP Engineer",
    "Senior Applied Scientist",
    "Backend Engineer",
    "Data Engineer",
    "Software Engineer",
}

medium_title_matches = {
    "DevOps Engineer",
    "Cloud Engineer",
    "Java Developer",
    "Full Stack Developer",
    ".NET Developer",
    "Mobile Developer",
    "QA Engineer",
}

weak_title_matches = {
    "Business Analyst",
    "Project Manager",
    "Operations Manager",
}

very_weak_title_matches = {
    "Accountant",
    "HR Manager",
    "Marketing Manager",
    "Graphic Designer",
    "Content Writer",
    "Customer Support",
    "Mechanical Engineer",
    "Civil Engineer",
}

title_score_map = {
    **{title: 1.0 for title in strong_title_matches},
    **{title: 0.7 for title in medium_title_matches},
    **{title: 0.3 for title in weak_title_matches},
    **{title: 0.0 for title in very_weak_title_matches},
}

df["title_score"] = df["profile_current_title"].map(title_score_map).fillna(0.0)

df["title_score"].value_counts()

In [ ]:
behavior_cols = [
    "redrob_signals_open_to_work_flag",
    "redrob_signals_recruiter_response_rate",
    "redrob_signals_notice_period_days",
    "redrob_signals_interview_completion_rate",
    "redrob_signals_offer_acceptance_rate",
]

df[behavior_cols].describe(include="all")

In [ ]:
def score_notice_period(days):
    if days <= 30:
        return 1.0
    if days <= 60:
        return 0.7
    if days <= 90:
        return 0.4
    return 0.1

df["open_to_work_score"] = df["redrob_signals_open_to_work_flag"].astype(float)
df["notice_score"] = df["redrob_signals_notice_period_days"].apply(score_notice_period)
df["offer_acceptance_score"] = df["redrob_signals_offer_acceptance_rate"].replace(-1, 0.5)

behavior_score_cols = [
    "open_to_work_score",
    "redrob_signals_recruiter_response_rate",
    "notice_score",
    "redrob_signals_interview_completion_rate",
    "offer_acceptance_score",
]

df["behavior_score"] = df[behavior_score_cols].mean(axis=1)

df["behavior_score"].describe()

In [ ]:
print(type(df.loc[0, "profile_summary"]))
print(type(df.loc[0, "career_history"]))
print(df.loc[0, "career_history"][0]["description"][:300])

In [ ]:
def safe_text(value):
    if value is None:
        return ""
    if not isinstance(value, (list, dict)) and pd.isna(value):
        return ""
    return str(value).strip()


def career_descriptions(career_history):
    if not isinstance(career_history, list):
        return ""
    return " ".join(
        safe_text(role.get("description"))
        for role in career_history
        if isinstance(role, dict) and safe_text(role.get("description"))
    )


df["candidate_text"] = (
    df["profile_summary"].apply(safe_text)
    + " "
    + df["profile_headline"].apply(safe_text)
    + " "
    + df["career_history"].apply(career_descriptions)
).str.strip()

sample_text = df.sample(3, random_state=42)[["candidate_id", "candidate_text"]].copy()
sample_text["candidate_text"] = sample_text["candidate_text"].str[:500]

print(sample_text.to_string(index=False))

In [ ]:
import re

keyword_groups = {
    "retrieval_keyword_count": [
        "retrieval",
        "retrieval system",
        "retrieval systems",
        "search",
        "semantic search",
        "vector search",
        "rag",
    ],
    "embedding_keyword_count": [
        "embedding",
        "embeddings",
        "semantic matching",
        "sentence transformer",
        "sentence transformers",
        "similarity search",
    ],
    "vector_db_keyword_count": [
        "vector database",
        "vector databases",
        "vector db",
        "vector store",
        "vector stores",
        "faiss",
        "pinecone",
        "weaviate",
        "milvus",
        "chroma",
        "qdrant",
    ],
    "llm_keyword_count": [
        "llm",
        "llms",
        "large language model",
        "large language models",
        "gpt",
        "openai",
        "anthropic",
        "langchain",
        "llamaindex",
        "prompt engineering",
    ],
    "evaluation_keyword_count": [
        "evaluation",
        "evaluations",
        "eval",
        "evals",
        "benchmark",
        "benchmarks",
        "metric",
        "metrics",
        "relevance scoring",
        "ranking quality",
    ],
}


def count_keywords(text, keywords):
    text = safe_text(text)
    return sum(
        len(re.findall(rf"(?<!\w){re.escape(keyword)}(?!\w)", text, flags=re.IGNORECASE))
        for keyword in keywords
    )


for count_col, keywords in keyword_groups.items():
    df[count_col] = df["candidate_text"].apply(lambda text: count_keywords(text, keywords))

keyword_count_cols = list(keyword_groups)
df["total_keyword_count"] = df[keyword_count_cols].sum(axis=1)

top_keyword_candidates = df.sort_values(
    ["total_keyword_count", "candidate_id"],
    ascending=[False, True],
).head(10)

top_keyword_candidates[["candidate_id", *keyword_count_cols, "total_keyword_count"]]


In [ ]:
top_ids = [
    "CAND_0000031",
    "CAND_0000021",
    "CAND_0000023"
]

for cid in top_ids:
    row = df[df["candidate_id"] == cid].iloc[0]

    print("=" * 100)
    print("Candidate:", cid)
    print("Title:", row["profile_current_title"])
    print("Experience:", row["profile_years_of_experience"])
    print("Company:", row["profile_current_company"])
    print()
    print(row["candidate_text"][:2000])
    print("\n")

In [ ]:
import re

strong_production_signals = [
    "shipped",
    "deployed",
    "in production",
    "production",
    "launched",
    "owned",
    "led",
    "designed",
    "built",
]

medium_production_signals = [
    "implemented",
    "developed",
    "worked on",
    "created",
]


def count_signal_occurrences(text, signals):
    text = safe_text(text)
    return sum(
        len(re.findall(rf"(?<!\w){re.escape(signal)}(?!\w)", text, flags=re.IGNORECASE))
        for signal in signals
    )


production_raw_score = (
    2 * df["candidate_text"].apply(lambda text: count_signal_occurrences(text, strong_production_signals))
    + df["candidate_text"].apply(lambda text: count_signal_occurrences(text, medium_production_signals))
)

max_production_raw_score = production_raw_score.max()
df["production_score"] = production_raw_score / max_production_raw_score if max_production_raw_score else 0.0

df.sort_values(
    ["production_score", "candidate_id"],
    ascending=[False, True],
).head(10)[["candidate_id", "production_score"]]


In [ ]:
weighted_score = (
    5 * df["retrieval_keyword_count"]
    + 4 * df["embedding_keyword_count"]
    + 4 * df["vector_db_keyword_count"]
    + 4 * df["evaluation_keyword_count"]
    + 2 * df["llm_keyword_count"]
)

max_weighted_score = weighted_score.max()
df["retrieval_score"] = weighted_score / max_weighted_score if max_weighted_score else 0.0

df.sort_values(
    ["retrieval_score", "candidate_id"],
    ascending=[False, True],
).head(10)[["candidate_id", "retrieval_score"]]


In [ ]:
df["final_score"] = (
    0.20 * df["experience_score"]
    + 0.15 * df["title_score"]
    + 0.15 * df["behavior_score"]
    + 0.25 * df["production_score"]
    + 0.25 * df["retrieval_score"]
)
top_candidates = df.sort_values(
    "final_score",
    ascending=False
)

top_candidates[
    [
        "candidate_id",
        "profile_current_title",
        "profile_years_of_experience",
        "final_score"
    ]
].head(20)

top_candidates[
    [
        "candidate_id",
        "profile_current_title",
        "profile_years_of_experience",
        "final_score"
    ]
].head(10)

In [ ]:
candidate = df[df["candidate_id"] == "CAND_0000036"]

candidate[
[
    "profile_current_title",
    "experience_score",
    "title_score",
    "behavior_score",
    "production_score",
    "retrieval_score",
    "final_score"
]
]
row = df[df["candidate_id"] == "CAND_0000036"].iloc[0]

print(row["candidate_text"][:2000])

In [ ]:
import re

technical_actions = [
    "built",
    "shipped",
    "deployed",
    "implemented",
    "designed",
    "developed",
]

technical_objects = [
    "model",
    "models",
    "ranking",
    "retrieval",
    "recommendation",
    "search",
    "pipeline",
    "pipelines",
    "embeddings",
    "vector database",
    "vector db",
    "ml system",
    "machine learning",
    "feature store",
    "xgboost",
    "lightgbm",
    "spark",
    "airflow",
    "faiss",
    "pinecone",
    "qdrant",
    "elasticsearch",
]

action_pattern = r"(?:" + "|".join(re.escape(action) for action in technical_actions) + r")"
object_pattern = r"(?:" + "|".join(re.escape(obj) for obj in technical_objects) + r")"
nearby_words = r"(?:\W+\w+){0,8}?\W+"

technical_production_pattern = re.compile(
    rf"(?<!\w)(?:{action_pattern}{nearby_words}{object_pattern}|{object_pattern}{nearby_words}{action_pattern})(?!\w)",
    flags=re.IGNORECASE,
)


def count_technical_production_phrases(text):
    return len(technical_production_pattern.findall(safe_text(text)))


technical_production_raw_score = df["candidate_text"].apply(count_technical_production_phrases)
max_technical_production_raw_score = technical_production_raw_score.max()
df["technical_production_score"] = (
    technical_production_raw_score / max_technical_production_raw_score
    if max_technical_production_raw_score
    else 0.0
)

df.sort_values(
    ["technical_production_score", "candidate_id"],
    ascending=[False, True],
).head(10)[["candidate_id", "technical_production_score"]]


In [ ]:
df["final_score_v2"] = (
    0.20 * df["experience_score"]
    + 0.20 * df["title_score"]
    + 0.10 * df["behavior_score"]
    + 0.10 * df["production_score"]
    + 0.15 * df["technical_production_score"]
    + 0.25 * df["retrieval_score"]
)

top_candidates_v2 = df.sort_values(
    "final_score_v2",
    ascending=False
)

top_candidates_v2[
[
    "candidate_id",
    "profile_current_title",
    "profile_years_of_experience",
    "final_score_v2"
]
].head(15)

In [ ]:
cid = "CAND_0000060"

row = df[df["candidate_id"] == cid]

row[
[
    "profile_current_title",
    "experience_score",
    "title_score",
    "behavior_score",
    "production_score",
    "technical_production_score",
    "retrieval_score",
    "final_score_v2"
]
]
# row.iloc[0]["candidate_text"][:1500]
row = df[df["candidate_id"] == "CAND_0000010"].iloc[0]

print("TITLE:", row["profile_current_title"])
print("EXP:", row["profile_years_of_experience"])
print()
print(row["candidate_text"][:2000])

In [ ]:
top20 = df.sort_values("final_score_v2", ascending=False)

top20[
    [
        "candidate_id",
        "profile_current_title",
        "profile_years_of_experience",
        "experience_score",
        "title_score",
        "behavior_score",
        "technical_production_score",
        "retrieval_score",
        "final_score_v2"
    ]
].head(20)

In [ ]:
bottom20 = df.sort_values(
    "final_score_v2",
    ascending=True
)

bottom20[
    [
        "candidate_id",
        "profile_current_title",
        "profile_years_of_experience",
        "final_score_v2"
    ]
].head(20)

In [ ]:
top100 = df.sort_values(
    "final_score_v2",
    ascending=False
).head(100)

top100["profile_current_title"].value_counts().head(20)

In [ ]:
import pandas as pd

top = pd.read_csv("../outputs/top_candidates.csv")


# Show summary statistics of final scores
# print(top["final_score"].describe())
print(top["profile_current_title"].value_counts())